# Phase E (clean) — Robustness: re-split stability + feature-family migration
**Self-contained. Just press Runtime -> Run all.** No dependence on other notebooks' session state.

1. Re-split robustness (different split seed) — does the prune80 per-class collapse reproduce?
2. Feature-family migration — which feature families lose SHAP importance under pruning.

Honest scope: section 1 is a split-*stability* check, not temporal/deployment realism (parquet was shuffled in Phase A; true temporal split is journal scope).

## 1. Setup (imports + paths) — run first

In [1]:
import os, gc, random, re
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

def seed_everything(s):
    os.environ['PYTHONHASHSEED']=str(s); random.seed(s); np.random.seed(s); tf.random.set_seed(s)
seed_everything(0)

ROOT='/content/drive/MyDrive/Trustworthiness_IoT'
if not os.path.isdir(f'{ROOT}/data'):
    from google.colab import drive; drive.mount('/content/drive')
CACHE=f'{ROOT}/data/cache'; RES=f'{ROOT}/results'; SH=f'{ROOT}/results/shap_v2'
CLASSES=['Benign','DDoS','DoS','Mirai','Recon','Spoofing','Web','BruteForce']
class_to_idx={c:i for i,c in enumerate(CLASSES)}; N_CLASSES=len(CLASSES)
print('TF', tf.__version__, '| setup OK')

Mounted at /content/drive
TF 2.20.0 | setup OK


## 2. Load CICIoT2023 (cached parquet)

In [2]:
df=pd.read_parquet(f'{CACHE}/ciciot_processed.parquet')
FEATURES=[c for c in df.columns if c!='y']
Xall=df[FEATURES].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan).fillna(0.0).astype('float32')
y=df['y'].map(class_to_idx).astype('int64').values
del df; gc.collect()
N_FEATURES=Xall.shape[1]
print('rows', len(y), '| features', N_FEATURES)

rows 46686579 | features 46


## 3. Re-split robustness check (split seed 7)

In [3]:
RESPLIT_SEED=7
idx=np.arange(len(y))
tr,tmp=train_test_split(idx,test_size=0.30,stratify=y,random_state=RESPLIT_SEED)
va,te=train_test_split(tmp,test_size=0.50,stratify=y[tmp],random_state=RESPLIT_SEED)
sc=StandardScaler().fit(Xall.iloc[tr])
Xtr=sc.transform(Xall.iloc[tr]).astype('float32'); ytr=y[tr]
Xva=sc.transform(Xall.iloc[va]).astype('float32'); yva=y[va]
Xte=sc.transform(Xall.iloc[te]).astype('float32'); yte=y[te]
bal=compute_class_weight('balanced',classes=np.arange(N_CLASSES),y=ytr)
tw=np.sqrt(bal); tw=tw/tw.mean(); cw={i:float(w) for i,w in enumerate(tw)}
print('re-split tr/va/te', len(tr), len(va), len(te))

re-split tr/va/te 32680605 7002987 7002987


In [4]:
def build_mlp(nf,nc,seed=0):
    seed_everything(seed); init=tf.keras.initializers.GlorotUniform(seed=seed)
    m=tf.keras.Sequential([tf.keras.layers.Input(shape=(nf,)),
        tf.keras.layers.Dense(128,activation='relu',kernel_initializer=init),
        tf.keras.layers.BatchNormalization(), tf.keras.layers.Dropout(0.3,seed=seed),
        tf.keras.layers.Dense(64,activation='relu',kernel_initializer=init),
        tf.keras.layers.Dropout(0.2,seed=seed),
        tf.keras.layers.Dense(nc,activation='softmax',kernel_initializer=init)])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    return m

seed_everything(0); M0d=build_mlp(N_FEATURES,N_CLASSES,0)
M0d.fit(Xtr,ytr,validation_data=(Xva,yva),epochs=40,batch_size=8192,class_weight=cw,
        callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)],verbose=2)
print('re-split M0 trained')

Epoch 1/40
3990/3990 - 28s - 7ms/step - accuracy: 0.8768 - loss: 0.0424 - val_accuracy: 0.9638 - val_loss: 0.1124
Epoch 2/40
3990/3990 - 17s - 4ms/step - accuracy: 0.9521 - loss: 0.0261 - val_accuracy: 0.9833 - val_loss: 0.0594
Epoch 3/40
3990/3990 - 17s - 4ms/step - accuracy: 0.9593 - loss: 0.0239 - val_accuracy: 0.9871 - val_loss: 0.0482
Epoch 4/40
3990/3990 - 17s - 4ms/step - accuracy: 0.9635 - loss: 0.0227 - val_accuracy: 0.9881 - val_loss: 0.0371
Epoch 5/40
3990/3990 - 17s - 4ms/step - accuracy: 0.9725 - loss: 0.0205 - val_accuracy: 0.9885 - val_loss: 0.0379
Epoch 6/40
3990/3990 - 17s - 4ms/step - accuracy: 0.9662 - loss: 0.0218 - val_accuracy: 0.9512 - val_loss: 0.1152
Epoch 7/40
3990/3990 - 17s - 4ms/step - accuracy: 0.9736 - loss: 0.0199 - val_accuracy: 0.9865 - val_loss: 0.0374
Epoch 8/40
3990/3990 - 16s - 4ms/step - accuracy: 0.9770 - loss: 0.0192 - val_accuracy: 0.9505 - val_loss: 0.1161
Epoch 9/40
3990/3990 - 17s - 4ms/step - accuracy: 0.9744 - loss: 0.0196 - val_accuracy: 

In [5]:
# prune the re-split M0 to 80% and compare per-class gap
def mmask(model,sp):
    M={}
    for i,l in enumerate(model.layers):
        if isinstance(l,tf.keras.layers.Dense):
            W=l.get_weights()[0]; M[i]=(np.abs(W)>=np.quantile(np.abs(W),sp)).astype('float32')
    return M
def amask(model,M):
    for i,m in M.items():
        w=model.layers[i].get_weights(); w[0]=w[0]*m; model.layers[i].set_weights(w)

seed_everything(0); M80=tf.keras.models.clone_model(M0d); M80.set_weights(M0d.get_weights())
M80.compile(optimizer=tf.keras.optimizers.Adam(5e-4),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
Mk=mmask(M80,0.80); amask(M80,Mk)
class RM(tf.keras.callbacks.Callback):
    def on_train_batch_end(self,b,logs=None): amask(self.model,Mk)
M80.fit(Xtr,ytr,validation_data=(Xva,yva),epochs=2,batch_size=8192,callbacks=[RM()],verbose=2); amask(M80,Mk)

def gap(conf,corr): return float(conf.mean()-corr.mean())
rows=[]
for name,m in [('M0_resplit',M0d),('M2_prune80_resplit',M80)]:
    P=m.predict(Xte,batch_size=8192,verbose=0); pred=P.argmax(1); conf=P.max(1); corr=(pred==yte).astype(float)
    d={'model':name,'overall_gap':round(gap(conf,corr),4)}
    for ci,c in enumerate(CLASSES):
        mm=(yte==ci); d[c]=round(gap(conf[mm],corr[mm]),4) if mm.sum() else np.nan
    rows.append(d)
dep=pd.DataFrame(rows); pd.set_option('display.width',240)
os.makedirs(RES,exist_ok=True); dep.to_csv(f'{RES}/resplit_calibration.csv',index=False)
print('===== RE-SPLIT (seed 7) PER-CLASS GAP =====')
print(dep.to_string(index=False))
print('\\nCompare to Phase B primary-split prune80 gaps: if Web/BruteForce/Recon still inflate, finding is split-stable.')

Epoch 1/2
3990/3990 - 43s - 11ms/step - accuracy: 0.8842 - loss: 0.2707 - val_accuracy: 0.9404 - val_loss: 0.1467
Epoch 2/2
3990/3990 - 34s - 8ms/step - accuracy: 0.9431 - loss: 0.1450 - val_accuracy: 0.9771 - val_loss: 0.0783
===== RE-SPLIT (seed 7) PER-CLASS GAP =====
             model  overall_gap  Benign    DDoS     DoS   Mirai  Recon  Spoofing    Web  BruteForce
        M0_resplit      -0.0036  -0.146 -0.0014  0.0004 -0.0006 0.0738    0.0084 0.1659      0.2683
M2_prune80_resplit      -0.0221  -0.399 -0.0131 -0.0718  0.0066 0.3430    0.5575 0.5849      0.5888
\nCompare to Phase B primary-split prune80 gaps: if Web/BruteForce/Recon still inflate, finding is split-stable.


## 4. Feature-family migration (from cached Phase C SHAP)

In [10]:
def family(name):
    f=str(name).lower()
    if any(k in f for k in ['header','tot size','tot sum','avg','min','max','std',
                            'magnitue','magnitude','radius','covariance','variance','weight','size']):
        return 'size'
    if any(k in f for k in ['duration','rate','srate','drate','iat','flow_dur','time']):
        return 'timing'
    if any(k in f for k in ['flag','syn','ack','fin','rst','psh','urg','cwr','ece']):
        return 'flags'
    if any(k in f for k in ['http','https','dns','telnet','smtp','ssh','irc','tcp','udp',
                            'dhcp','arp','icmp','ipv','llc','protocol']):
        return 'protocol'
    if any(k in f for k in ['count','number']):
        return 'volume'
    return 'other'

fams=[family(f) for f in FEATURES]; FAMSET=sorted(set(fams))
def share(A):
    imp=np.abs(A).mean(0); s={fa:0.0 for fa in FAMSET}
    for j,fa in enumerate(fams): s[fa]+=imp[j]
    tot=sum(s.values()) or 1.0; return {k:v/tot for k,v in s.items()}
s0=share(A0); s80=share(A80)
mig=pd.DataFrame([{'family':fa,'n_feats':fams.count(fa),'M0_share':round(s0[fa],3),
                   'prune80_share':round(s80[fa],3),'delta':round(s80[fa]-s0[fa],3)} for fa in FAMSET]).sort_values('delta')
mig.to_csv(f'{RES}/feature_family_migration.csv',index=False)
print(mig.to_string(index=False))
print('\nfamily counts:', {fa:fams.count(fa) for fa in FAMSET})

  family  n_feats  M0_share  prune80_share  delta
    size       12     0.396          0.365 -0.030
protocol       15     0.199          0.183 -0.016
   flags       12     0.132          0.123 -0.008
  volume        1     0.080          0.102  0.023
  timing        6     0.194          0.226  0.032

family counts: {'flags': 12, 'protocol': 15, 'size': 12, 'timing': 6, 'volume': 1}


In [11]:
A0=np.load(f'{SH}/M0_s0_es0.npy'); A80=np.load(f'{SH}/M2_prune80_es0.npy')
assert A0.shape[1]==len(FEATURES)==A80.shape[1], f'mismatch {A0.shape[1]} vs {len(FEATURES)}'

def family(name):
    f=str(name).lower()
    if any(k in f for k in ['time','duration','iat','rate','flow_dur','idle','active']): return 'timing'
    if any(k in f for k in ['len','size','byte','header','payload','tot_size','avg']): return 'size'
    if any(k in f for k in ['flag','syn','ack','fin','rst','psh','urg','cwr','ece']): return 'flags'
    if any(k in f for k in ['proto','tcp','udp','icmp','http','dns','ssh','telnet','irc','smtp','arp']): return 'protocol'
    if any(k in f for k in ['count','number','sum','num','cnt','variance','std','tot']): return 'volume'
    return 'other'
fams=[family(f) for f in FEATURES]; FAMSET=sorted(set(fams))

def share(A):
    imp=np.abs(A).mean(0); s={fa:0.0 for fa in FAMSET}
    for j,fa in enumerate(fams): s[fa]+=imp[j]
    tot=sum(s.values()) or 1.0; return {k:v/tot for k,v in s.items()}
s0=share(A0); s80=share(A80)
mig=pd.DataFrame([{'family':fa,'n_feats':fams.count(fa),'M0_share':round(s0[fa],3),
                   'prune80_share':round(s80[fa],3),'delta':round(s80[fa]-s0[fa],3)} for fa in FAMSET]).sort_values('delta')
mig.to_csv(f'{RES}/feature_family_migration.csv',index=False)
print('===== FEATURE-FAMILY MIGRATION =====')
print(mig.to_string(index=False))
print('\\nfamily counts:', {fa:fams.count(fa) for fa in FAMSET})

===== FEATURE-FAMILY MIGRATION =====
  family  n_feats  M0_share  prune80_share  delta
    size        3     0.112          0.062 -0.050
protocol       12     0.197          0.182 -0.014
   flags       12     0.132          0.123 -0.008
  volume        5     0.129          0.131  0.002
  timing        6     0.194          0.226  0.032
   other        8     0.236          0.275  0.039
\nfamily counts: {'flags': 12, 'other': 8, 'protocol': 12, 'size': 3, 'timing': 6, 'volume': 5}


In [7]:
# Full feature -> family map (paste this back so keywords can be tuned if needed)
print('===== FEATURE -> FAMILY MAP =====')
for f,fa in zip(FEATURES,fams): print(f'{f:<32} -> {fa}')

===== FEATURE -> FAMILY MAP =====
flow_duration                    -> timing
Header_Length                    -> size
Protocol Type                    -> protocol
Duration                         -> timing
Rate                             -> timing
Srate                            -> timing
Drate                            -> timing
fin_flag_number                  -> flags
syn_flag_number                  -> flags
rst_flag_number                  -> flags
psh_flag_number                  -> flags
ack_flag_number                  -> flags
ece_flag_number                  -> flags
cwr_flag_number                  -> flags
ack_count                        -> flags
syn_count                        -> flags
fin_count                        -> flags
urg_count                        -> flags
rst_count                        -> flags
HTTP                             -> protocol
HTTPS                            -> protocol
DNS                              -> protocol
Telnet                    

## 5. Commit (run manually after numbers look right)

In [8]:
# !git -C "{ROOT}" add results/ notebooks/
# !git -C "{ROOT}" commit -q -m "Phase E: re-split robustness + feature-family migration"
# !git -C "{ROOT}" push origin main
print('results now in', RES)
print([f for f in os.listdir(RES) if f.endswith('.csv')])

results now in /content/drive/MyDrive/Trustworthiness_IoT/results
['deployability_proxy.csv', 'calibration_perclass.csv', 'calibration_test.csv', 'faithfulness_agreement.csv', 'faithfulness_perclass_spearman.csv', 'faithfulness_perturbation.csv', 'faithfulness_agreement_v2.csv', 'faithfulness_perclass_spearman_v2.csv', 'faithfulness_perturbation_v2.csv', 'resplit_calibration.csv', 'feature_family_migration.csv']


In [12]:
ROOT='/content/drive/MyDrive/Trustworthiness_IoT'
!git -C "{ROOT}" add results/ notebooks/
!git -C "{ROOT}" commit -q -m "Phase E: re-split robustness (split-stable) + feature-family migration (timing up, size down)"
!git -C "{ROOT}" push origin main
print('pushed')

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 8 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 11.69 KiB | 1.67 MiB/s, done.
Total 6 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/anasbiswas1/trustworthiness-under-compression.git
   e82ffd6..c8cb345  main -> main
pushed


## What to paste back
1. The **RE-SPLIT per-class gap** table.
2. The **FEATURE-FAMILY MIGRATION** table + `family counts`.
3. The **FEATURE -> FAMILY MAP**.

If many features land in `other`, paste the map and the keyword lists get tuned to CICIoT2023's real names.